This notebook creates all plots regarding results on datasets containing dose-response curves on dimeric and monomeric IL-10 variants, including direct dose-response curves and violin plots on the amplitude and EC50. The notebook contains simulations on:

- Control and Receptor Memory models on dimeric IL-10 variants with increased IL-10RB binding (Figures 3A-E and EV3A-E, Appendix Figures S4 and S6)
- Receptor Memory model on monomeric IL-10 variants with increased IL-10RB binding (Figure 3F, Appendix Figure S10)
- Receptor Memory model on single-chain IL-10 variants (Figure 3G, Appendix Figure S11)
- Receptor memory model assumming STAT1 can also be phophorylated in cis by Jak1 (Appendix Figure S8)
- Receptor memory model explaining signaling decay at high IL-10 concentrations through IL-10RA binding producing half-saturation (Appendix Figures S9E-H)
- Receptor memory model explaining signaling decay at high IL-10 concentrations through higher IL-10RB binding (Appendix Figures S9I-L)
- Receptor memory model, but changing the phosphorylation rate and see its effect in an IL-10RB mutant (Appendix Figure S12A)
- Receptor memory model, but see the effects of chaging the Kon vs Koff in an IL-10RB mutant (Appendix Figure S12B)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as Patch
from matplotlib import use
use('Agg')
import warnings
warnings.filterwarnings('ignore')
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
import seaborn as sns
from scipy.optimize import curve_fit
from src.simulate_func import hill_func

In [2]:
# Colors for mutants:
color_sim = ["blueviolet"]
color_invit = ["purple"]
color_area = ["violet"]
markers = ['o','^']
linestyles = ["dotted"]

### Fitted parameter set dose-response curves in the Control and simplifed and full Receptor memory model

In [3]:
# Normalize in each plot for the maximum response of the WT IL-10
sim_data_list = ["simulations_fit_IL10_Control_eIC.csv","simulations_fit_IL10_RAp_eIC.csv","simulations_fit_IL10_RAp_ODE_eIC.csv"]
plots_folder_list = ["IL10","IL10_RAp","IL10_RAp_ODE"]

df_invit = pd.concat([pd.read_csv('data/signaling/IL10_STAT_data.tsv.gz', sep="\t", compression="gzip"), pd.read_csv('data/signaling/IL10_STAT_data_CD8_Gorby.tsv.gz', sep="\t", compression="gzip")], ignore_index=True)
df_invit.loc[len(df_invit)] = [12, 'T8.Mean', 'WT', 'pSTAT1', 1e-7, 100.297619047619]
df_invit.loc[len(df_invit)] = [12, 'T8.Mean', 'Super-10', 'pSTAT1', 1e-7, 146.130952380952]
df_invit.loc[len(df_invit)] = [13, 'T4.Mean', 'WT', 'pSTAT1', 1e-7, 100.595238095238]
df_invit.loc[len(df_invit)] = [13, 'T4.Mean', 'Super-10', 'pSTAT1', 1e-7, 136.309523809524]

for sim_data_i in range(0, len(sim_data_list)):
    df_sim = pd.read_csv("results/fit_param/"+sim_data_list[sim_data_i])
    # Normalize data to the maximum WT response
    for plot_num in df_sim["Plot"].drop_duplicates():
        df_sim.loc[(df_sim["Plot"]==plot_num),"Result"] = df_sim.loc[(df_sim["Plot"]==plot_num),"Result"]/df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"].max()*100
    # Plot dose-response curves
    for plot_num in df_sim["Plot OG"].drop_duplicates():
        df_invit_plot = df_invit.loc[df_invit["Plot"]==plot_num]
    
        # First plot WT
        variant = "WT"
        variant_list = df_sim.loc[df_sim["Plot OG"]==plot_num,"Variant"].drop_duplicates().to_list()
        variant_list.remove("WT")
        fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
        IL = df_sim.loc[(df_sim["Plot OG"]==plot_num),"IL0"].drop_duplicates().values
        responses = df_sim.loc[(df_sim["Plot OG"]==plot_num)&(df_sim["Variant"]==variant)][["Plot","IL0","Result"]].pivot(index='IL0', columns='Plot', values='Result').transpose().values
        original_response = responses[0]
        low_response = np.percentile(responses, 5, axis=0)
        high_response = np.percentile(responses, 95, axis=0)
    
        ax.fill_between(IL, low_response, high_response, color="lightgreen", alpha=0.5)
        ax.plot(IL, original_response, linewidth=7.5, label=variant,color="darkgreen", linestyle="dashed")
        ax.scatter(df_invit_plot.loc[df_invit_plot["Variant"]==variant]["IL"].values, df_invit_plot.loc[df_invit_plot["Variant"]==variant]["pSTAT"].values, s=200, linewidth=2, color="forestgreen",marker='*')
        
        i=0
        # Then plot the other variants
        for variant in variant_list:
            responses = df_sim.loc[(df_sim["Plot OG"]==plot_num)&(df_sim["Variant"]==variant)][["Plot","IL0","Result"]].pivot(index='IL0', columns='Plot', values='Result').transpose().values
            original_response = responses[0]
            low_response = np.percentile(responses, 5, axis=0)
            high_response = np.percentile(responses, 95, axis=0)
    
            ax.fill_between(IL, low_response, high_response, color=color_area[i], alpha=0.4)
            ax.plot(IL, original_response, linewidth=7.5, label=variant,color=color_sim[i], linestyle=linestyles[i])
            ax.scatter(df_invit_plot.loc[df_invit_plot["Variant"]==variant]["IL"].values, df_invit_plot.loc[df_invit_plot["Variant"]==variant]["pSTAT"].values, s=200, linewidth=2, color=color_invit[i],marker=markers[i])
            i += 1
        plt.legend(loc="upper left", fontsize=20)
        plt.xscale('log')
        ax.set_xticks(10**np.arange(round(np.log10(df_sim.loc[df_sim["Plot OG"]==plot_num,"IL0"].min()),0)+1,round(np.log10(df_sim.loc[df_sim["Plot OG"]==plot_num,"IL0"].max()),0)+1,2))
        ax.spines["bottom"].set_linewidth(4)
        ax.spines["left"].set_linewidth(4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_tick_params(width=5, length=10)
        ax.yaxis.set_tick_params(width=5, length=10)
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=25)
        cell = df_sim.loc[df_sim["Plot OG"]==plot_num,"Cell_type"].values[0]
        plt.title(cell, fontsize=25)
        plt.ylabel(list(df_sim.loc[df_sim["Plot OG"]==plot_num,"STAT_type"])[0] +' (%)', fontsize=25)
        plt.xlabel('IL-10 (M)', fontsize=25)
        plt.savefig('figures/fit_param/'+plots_folder_list[sim_data_i]+'/Plot_n'+str(df_sim.loc[df_sim["Plot OG"]==plot_num,"Plot"].values[0])+'_cell_'+df_sim.loc[df_sim["Plot OG"]==plot_num,"Cell_type"].values[0]+'.pdf', transparent=True, bbox_inches="tight")
        plt.show()

### Violin plots of dose-response curves' amplitude in the Control and simplifed and full Receptor memory model

In [4]:
# Get experimental data on response amplitude in the different cells
df_EC50_AMP = pd.concat([pd.read_csv('data/signaling/IL10_EC50_AMP.tsv.gz', sep="\t", compression="gzip"), pd.read_csv('data/signaling/IL10_EC50_AMP_CD8_Gorby.tsv.gz', sep="\t", compression="gzip")], ignore_index=True)
variants = ["WT","Super-10","R5A11D"]
df_EC50_AMP = df_EC50_AMP.loc[(df_EC50_AMP["Variant"].isin(variants))&(df_EC50_AMP["Plot"].isin([1,2,8,9,12,13,14,15]))]
df_EC50_AMP = df_EC50_AMP.loc[df_EC50_AMP["Variant"] != "WT"]
identifier = df_EC50_AMP["Cell_type"]+"__"+df_EC50_AMP["Variant"]+"__"+df_EC50_AMP["STAT_type"]
df_EC50_AMP = df_EC50_AMP[["Amp"]].transpose()
df_EC50_AMP.columns = identifier
df_EC50_AMP.columns = [col.replace("MO.Mean","Monocytes") for col in df_EC50_AMP.columns]
df_EC50_AMP.columns = [col.replace("T4.Mean","CD4+ T cells") for col in df_EC50_AMP.columns]
df_EC50_AMP.columns = [col.replace("T8.Mean","CD8+ T cells") for col in df_EC50_AMP.columns]
df_EC50_AMP = df_EC50_AMP[['Monocytes__R5A11D__pSTAT3','Monocytes__R5A11D__pSTAT1','CD8+ T cells__R5A11D__pSTAT3','CD8+ T cells__R5A11D__pSTAT1','CD4+ T cells__Super-10__pSTAT3','CD8+ T cells__Super-10__pSTAT3','CD8+ T cells__Super-10__pSTAT1','CD4+ T cells__Super-10__pSTAT1']]

In [5]:
model_list = ["IL10_Control","IL10_RAp","IL10_RAp_ODE"]
for model in model_list:
    # Get distribution of simulated amplitude in the different cells
    df_sim = pd.read_csv("results/fit_param/simulations_fit_"+model+"_eIC.csv")
    df_sim = df_sim.loc[df_sim["Plot OG"].isin([1,2,8,9,12,13,14,15])]
    for plot_num in df_sim["Plot"].drop_duplicates():
        df_sim.loc[(df_sim["Plot"]==plot_num),"Result"] = df_sim.loc[(df_sim["Plot"]==plot_num),"Result"]/df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"].max()*100
        df_amp = pd.DataFrame()
    for plot_num_OG in df_sim["Plot OG"].drop_duplicates():
        df_amp[df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][0]+"__"+df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][1]+"__"+df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][2]] = [df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"] != "WT"),"Result"].max() for plot_num in df_sim.loc[(df_sim["Plot OG"]==plot_num_OG),"Plot"].drop_duplicates().values]
    df_amp.columns = [col.replace("MO.Mean","Monocytes") for col in df_amp.columns]
    df_amp.columns = [col.replace("T4.Mean","CD4+ T cells") for col in df_amp.columns]
    df_amp.columns = [col.replace("T8.Mean","CD8+ T cells") for col in df_amp.columns]
    df_amp = df_amp[['Monocytes__R5A11D__pSTAT3','Monocytes__R5A11D__pSTAT1','CD8+ T cells__R5A11D__pSTAT3','CD8+ T cells__R5A11D__pSTAT1','CD4+ T cells__Super-10__pSTAT3','CD8+ T cells__Super-10__pSTAT3','CD8+ T cells__Super-10__pSTAT1','CD4+ T cells__Super-10__pSTAT1']]
    
    # pSTAT1 amplitude
    df_amp_S1 = df_amp[[col for col in df_amp.columns if "pSTAT1" in col]]
    df_amp_S1.columns = [col.replace("__pSTAT1","").replace("__"," \n") for col in df_amp_S1.columns]
    df_EC50_AMP_S1 = df_EC50_AMP[[col for col in df_EC50_AMP.columns if "pSTAT1" in col]]
    df_EC50_AMP_S1.columns = [col.replace("__pSTAT1","").replace("__"," \n") for col in df_EC50_AMP_S1.columns]
    
    # pSTAT3 amplitude
    df_amp_S3 = df_amp[[col for col in df_amp.columns if "pSTAT3" in col]]
    df_amp_S3.columns = [col.replace("__pSTAT3","").replace("__"," \n") for col in df_amp_S3.columns]
    df_EC50_AMP_S3 = df_EC50_AMP[[col for col in df_EC50_AMP.columns if "pSTAT3" in col]]
    df_EC50_AMP_S3.columns = [col.replace("__pSTAT3","").replace("__"," \n") for col in df_EC50_AMP_S3.columns]
    df_EC50_AMP_S3[df_EC50_AMP_S3<100] = 100
    
    # Generate figure
    fig, ax = plt.subplots(1, 1, figsize=(13, 8), dpi=400)
    pos_base = np.arange(len(df_amp_S1.columns)) 
    offset = 0.2
    # Get the violin plots
    quartile1, medians, quartile3 = np.percentile(df_amp_S1.T, [25, 50, 75], axis=1)
    vp_S1 = ax.violinplot(df_amp_S1, positions=pos_base + offset,
                  widths=0.3, showextrema=False)
    for b in vp_S1['bodies']:
        b.set_facecolor("Darkred")
        b.set_edgecolor("black")
        b.set_alpha(1)
    ax.vlines(pos_base + offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
    ax.scatter(pos_base + offset, df_amp_S1.mean(), marker='o', color='white', s=40, zorder=9)
    ax.scatter(pos_base + offset,df_EC50_AMP_S1.values, s=250, marker='D', color="gold", edgecolor="black", linewidth=2, zorder=10)

    quartile1, medians, quartile3 = np.percentile(df_amp_S3.T, [25, 50, 75], axis=1)
    vp_S3 = ax.violinplot(df_amp_S3, positions=pos_base - offset,
                  widths=0.3, showextrema=False)
    for b in vp_S3['bodies']:
        b.set_facecolor("cornflowerblue")
        b.set_edgecolor("black")
        b.set_alpha(1)
    ax.vlines(pos_base - offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
    ax.scatter(pos_base - offset, df_amp_S3.mean(), marker='o', color='white', s=40, zorder=9)
    ax.scatter(pos_base - offset,df_EC50_AMP_S3.values, s=250, marker='D', color="gold", edgecolor="black", linewidth=2, zorder=10)

    # Customize appearance
    ax.set_xticks(pos_base)
    ax.set_xticklabels(df_EC50_AMP_S3.columns, fontsize=18)
    plt.ylabel("pSTAT (%)", fontsize=18)
    legend_handles = [
        Patch.Patch(facecolor="cornflowerblue", label="pSTAT3"),
        Patch.Patch(facecolor="Darkred", label="pSTAT1")
    ]
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    plt.ylabel("pSTAT (%)", fontsize=18)
    ax.legend(handles=legend_handles, fontsize=18,loc="upper left")
    plt.savefig('figures/fit_param/violin_amplitude_'+model+'.pdf', transparent=True, bbox_inches="tight")

### Fitted parameter set dose-response curves in the Receptor memory model in single-chain IL-10 variants

In [6]:
df_sim = pd.read_csv("results/fit_param/simulations_fit_IL10_RAp_ODE_scIL10.csv")
# Normalize data to the maximum WT response
for plot_num in df_sim["Plot"].drop_duplicates():
    df_sim.loc[(df_sim["Plot"]==plot_num),"Result"] = df_sim.loc[(df_sim["Plot"]==plot_num),"Result"]/df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"].max()*100
# Plot dose-response curves
for plot_num in df_sim["Plot OG"].drop_duplicates():
    # First plot WT
    variant = "WT"
    variant_list = df_sim.loc[df_sim["Plot OG"]==plot_num,"Variant"].drop_duplicates().to_list()
    variant_list.remove("WT")
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    IL = df_sim.loc[(df_sim["Plot OG"]==plot_num),"IL0"].drop_duplicates().values
    responses = df_sim.loc[(df_sim["Plot OG"]==plot_num)&(df_sim["Variant"]==variant)][["Plot","IL0","Result"]].pivot(index='IL0', columns='Plot', values='Result').transpose().values
    original_response = responses[0]
    low_response = np.percentile(responses, 5, axis=0)
    high_response = np.percentile(responses, 95, axis=0)
    ax.fill_between(IL, low_response, high_response, color="lightgreen", alpha=0.5)
    ax.plot(IL, original_response, linewidth=7.5, label=variant,color="darkgreen", linestyle="dashed")
    
    i=0
    # Then plot the other variants
    for variant in variant_list:
        responses = df_sim.loc[(df_sim["Plot OG"]==plot_num)&(df_sim["Variant"]==variant)][["Plot","IL0","Result"]].pivot(index='IL0', columns='Plot', values='Result').transpose().values
        original_response = responses[0]
        low_response = np.percentile(responses, 5, axis=0)
        high_response = np.percentile(responses, 95, axis=0)
        ax.fill_between(IL, low_response, high_response, color=color_area[i], alpha=0.4)
        ax.plot(IL, original_response, linewidth=7.5, label=variant,color=color_sim[i], linestyle=linestyles[i])
        i += 1
    plt.legend(loc="upper left", fontsize=20)
    plt.xscale('log')
    ax.set_xticks(10**np.arange(round(np.log10(df_sim.loc[df_sim["Plot OG"]==plot_num,"IL0"].min()),0)+1,round(np.log10(df_sim.loc[df_sim["Plot OG"]==plot_num,"IL0"].max()),0)+1,2))
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    cell = df_sim.loc[df_sim["Plot OG"]==plot_num,"Cell_type"].values[0]
    plt.title(cell, fontsize=25)
    plt.ylabel(list(df_sim.loc[df_sim["Plot OG"]==plot_num,"STAT_type"])[0] +' (%)', fontsize=25)
    plt.xlabel('IL-10 (M)', fontsize=25)
    plt.savefig('figures/fit_param/IL10_RAp_ODE/scIL10/Plot_n'+str(df_sim.loc[df_sim["Plot OG"]==plot_num,"Plot"].values[0])+'_cell_'+df_sim.loc[df_sim["Plot OG"]==plot_num,"Cell_type"].values[0]+'.pdf', transparent=True, bbox_inches="tight")
    plt.close()

### Violin plots of dose-response curves' EC50 (single chain IL-10)

In [7]:
# Get data from simulations
df_sim = pd.read_csv("results/fit_param/simulations_fit_IL10_RAp_ODE_scIL10.csv")
df_EC50_sim = df_sim[["Plot","Cell_type","Variant","Plot OG"]].drop_duplicates().reset_index(drop=True)
EC50_list = []
for plot_num,cell_type,variant in [df_EC50_sim.loc[i].values[:-1] for i in range(0,len(df_EC50_sim))]:
    df_var = df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Cell_type"]==cell_type)&(df_sim["Variant"]==variant)]
    fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values*100/(df_var["Result"].max()), bounds = ([95,df_var["IL0"].min()*10], [105, df_var["IL0"].max()/5]),p0=(100,1e-8))
    EC50_list.append(np.log10(fit[1]))
df_EC50_sim["EC50"] = EC50_list
df_EC50_sim["Cell_Variant"] = df_EC50_sim["Cell_type"] + "\n" + df_EC50_sim["Variant"]
df_EC50_sim["Marker"] = (df_EC50_sim["Plot"]-df_EC50_sim["Plot OG"])/2
df_EC50_sim = df_EC50_sim.pivot(index='Marker', columns='Cell_Variant', values='EC50').reset_index(drop=True)
df_EC50_sim.columns = df_EC50_sim.columns.values
df_EC50_sim = df_EC50_sim.where(df_EC50_sim.ge(df_EC50_sim.quantile(0.05)))

# Get experimental EC50s
df_EC50_AMP_train = pd.read_csv('data/signaling/IL10sc_EC50_AMP_Montero.tsv.gz', sep="\t", compression="gzip")
df_EC50_AMP_train["Plot__Variant"] = df_EC50_AMP_train["Plot"].astype(str)+"__"+df_EC50_AMP_train["Variant"]
df_EC50_AMP_train.index = df_EC50_AMP_train["Cell_type"] + "\n" + df_EC50_AMP_train["Variant"]
df_EC50_AMP_train = df_EC50_AMP_train[["EC50"]].transpose().reset_index(drop=True)
df_EC50_AMP_train = df_EC50_AMP_train[df_EC50_sim.columns]

In [8]:
# Generate figure
fig, ax = plt.subplots(1, 1, figsize=(7, 8), dpi=400)
pos_base = np.array([0,0.9])
offset = 0.18
# Get the violin plots
quartile1, medians, quartile3 = np.percentile([df_EC50_sim[cell].dropna() for cell in df_EC50_sim.columns if "MutSC1" in cell], [25, 50, 75], axis=1)
vp_S1 = ax.violinplot([df_EC50_sim[cell].dropna() for cell in df_EC50_sim.columns if "MutSC1" in cell], positions=pos_base + offset,
              widths=0.3, showextrema=False)
for b in vp_S1['bodies']:
    b.set_facecolor("purple")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base + offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base + offset, df_EC50_sim[[cell for cell in df_EC50_sim.columns if "MutSC1" in cell]].mean(), marker='o', color='white', s=40, zorder=9)
ax.scatter(pos_base + offset,df_EC50_AMP_train[[cell for cell in df_EC50_sim.columns if "MutSC1" in cell]].values, s=250, marker='D', color="gold", edgecolor="black", linewidth=2, zorder=10)

quartile1, medians, quartile3 = np.percentile([df_EC50_sim[cell].dropna() for cell in df_EC50_sim.columns if "WT" in cell], [25, 50, 75], axis=1)
vp_S3 = ax.violinplot([df_EC50_sim[cell].dropna() for cell in df_EC50_sim.columns if "WT" in cell], positions=pos_base - offset,
              widths=0.3, showextrema=False)
for b in vp_S3['bodies']:
    b.set_facecolor("darkgreen")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base - offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base - offset, df_EC50_sim[[cell for cell in df_EC50_sim.columns if "WT" in cell]].mean(), marker='o', color='white', s=40, zorder=9)
ax.scatter(pos_base - offset,df_EC50_AMP_train[[cell for cell in df_EC50_sim.columns if "WT" in cell]].values, s=250, marker='D', color="gold", edgecolor="black", linewidth=2, zorder=10)

# Customize appearance
ax.set_xticks(pos_base)
ax.set_xticklabels([cell.split("\n")[0] for cell in df_EC50_sim.columns if "MutSC1" in cell], fontsize=18)
plt.ylabel("pSTAT (%)", fontsize=18)
legend_handles = [
    Patch.Patch(facecolor="darkgreen", label="WT"),
    Patch.Patch(facecolor="purple", label="MutSC1")
]
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)
plt.ylabel("EC50 (M)", fontsize=18)
ax.legend(handles=legend_handles, fontsize=18,loc="upper left")
plt.savefig('figures/fit_param/violin_amplitude_scIL10_filt.pdf', transparent=True, bbox_inches="tight")
plt.close()
print("MAE WT and MutSC1 in BLaER1 and Haftl cell lines: "+str((df_EC50_sim-df_EC50_AMP_train.values).abs().melt()["value"].mean()))

MAE WT and MutSC1 in BLaER1 and Haftl cell lines: 0.32983189316642975


### Fitted parameter set dose-response curves (monomeric IL-10)

In [9]:
# Normalize in each plot for the maximum response of the WT IL-10
df_invit = pd.read_csv("data/signaling/IL10M_STAT_data_Gorby.tsv.gz", sep='\t', compression='gzip')
df_sim = pd.read_csv("results/fit_param/simulations_fit_IL10_RAp_ODE_MONO_fit_eIC.csv")
# Normalize data to the maximum WT response
for plot_num in df_sim["Plot"].drop_duplicates():
    df_sim.loc[(df_sim["Plot"]==plot_num),"Result"] = df_sim.loc[(df_sim["Plot"]==plot_num),"Result"]/df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"].max()*100

# For each plot get mean response and the 90% confidence intervals
for plot_num in df_sim["Plot OG"].drop_duplicates():
    df_invit_plot = df_invit.loc[df_invit["Plot"]==plot_num]

    # First plot WT
    variant = "WT"
    variant_list = df_sim.loc[df_sim["Plot OG"]==plot_num,"Variant"].drop_duplicates().to_list()
    variant_list.remove("WT")
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    IL = df_sim.loc[(df_sim["Plot OG"]==plot_num),"IL0"].drop_duplicates().values
    responses = df_sim.loc[(df_sim["Plot OG"]==plot_num)&(df_sim["Variant"]==variant)][["Plot","IL0","Result"]].pivot(index='IL0', columns='Plot', values='Result').transpose().values
    original_response = responses[0]
    low_response = np.percentile(responses, 5, axis=0)
    high_response = np.percentile(responses, 95, axis=0)

    ax.fill_between(IL, low_response, high_response, color='lightgreen', alpha=0.5)
    ax.plot(IL, original_response, linewidth=7.5, label=variant,color="darkgreen", linestyle="dashed")
    ax.scatter(df_invit_plot.loc[df_invit_plot["Variant"]==variant]["IL"].values, df_invit_plot.loc[df_invit_plot["Variant"]==variant]["pSTAT"].values, s=200, linewidth=2, color="forestgreen",marker='*')

    i=0
    # Then plot the other variants
    for variant in variant_list:
        responses = df_sim.loc[(df_sim["Plot OG"]==plot_num)&(df_sim["Variant"]==variant)][["Plot","IL0","Result"]].pivot(index='IL0', columns='Plot', values='Result').transpose().values
        original_response = responses[0]
        low_response = np.percentile(responses, 5, axis=0)
        high_response = np.percentile(responses, 95, axis=0)

        ax.fill_between(IL, low_response, high_response, color=color_area[i], alpha=0.4)
        ax.plot(IL, original_response, linewidth=7.5, label=variant,color=color_sim[i], linestyle=linestyles[i])
        ax.scatter(df_invit_plot.loc[df_invit_plot["Variant"]==variant]["IL"].values, df_invit_plot.loc[df_invit_plot["Variant"]==variant]["pSTAT"].values, s=200, linewidth=2, color=color_invit[i],marker=markers[i])

        i += 1
    plt.legend(loc="upper left", fontsize=20)
    plt.xscale('log')
    ax.set_xticks(10**np.arange(round(np.log10(df_sim.loc[df_sim["Plot OG"]==plot_num,"IL0"].min()),0)+1,round(np.log10(df_sim.loc[df_sim["Plot OG"]==plot_num,"IL0"].max()),0)+1,2))
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    cell = df_sim.loc[df_sim["Plot OG"]==plot_num,"Cell_type"].values[0]
    plt.title(cell, fontsize=25)
    plt.ylabel(list(df_sim.loc[df_sim["Plot OG"]==plot_num,"STAT_type"])[0] +' (%)', fontsize=25)
    plt.xlabel('IL-10 (M)', fontsize=25)
    plt.savefig('figures/fit_param/IL10_RAp_ODE_MONO/Plot_n'+str(df_sim.loc[df_sim["Plot OG"]==plot_num,"Plot"].values[0])+'_cell_'+df_sim.loc[df_sim["Plot OG"]==plot_num,"Cell_type"].values[0]+'.pdf', transparent=True, bbox_inches="tight")
    plt.show()

### Violin plots of dose-response curves' amplitude (monomeric IL-10)

In [10]:
# Get amplitude of in vitro dose response curves
df_sim = pd.read_csv("results/fit_param/simulations_fit_IL10_RAp_ODE_MONO_fit_eIC.csv")
df_invit = pd.read_csv("data/signaling/IL10M_STAT_data_Gorby.tsv.gz", sep='\t', compression='gzip')
df_EC50_AMP = df_invit[['Plot', 'Cell_type', 'Variant', 'STAT_type']].drop_duplicates()
df_EC50_AMP.reset_index(drop=True,inplace=True)
AMP_list = []
for plot_num,cell_type,variant in [df_EC50_AMP.loc[i].values[:-1] for i in range(0,len(df_EC50_AMP))]:
    df_var = df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Cell_type"]==cell_type)&(df_sim["Variant"]==variant)]
    max_WT = df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Cell_type"]==cell_type)&(df_sim["Variant"]=="WT"),"Result"].max()
    fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values*100/(max_WT), bounds = ([0,df_var["IL0"].min()/10], [df_var["Result"].max()*100/(max_WT)+15, df_var["IL0"].max()*10]))
    AMP_list.append(fit[0])
df_EC50_AMP["Amp"] = AMP_list 
df_EC50_AMP = df_EC50_AMP.loc[df_EC50_AMP["Variant"] != "WT"]
identifier = df_EC50_AMP["Cell_type"]+"__"+df_EC50_AMP["Variant"]+"__"+df_EC50_AMP["STAT_type"]
df_EC50_AMP = df_EC50_AMP[["Amp"]].transpose()
df_EC50_AMP.columns = identifier

# Get amplitudes of experimental data
df_EC50_AMP_true = df_invit[['Plot', 'Cell_type', 'Variant', 'STAT_type']].drop_duplicates()
df_EC50_AMP_true.reset_index(drop=True,inplace=True)
AMP_list = []
for plot_num,cell_type,variant in [df_EC50_AMP_true.loc[i].values[:-1] for i in range(0,len(df_EC50_AMP_true))]:
    df_var = df_invit.loc[(df_invit["Plot"]==plot_num)&(df_invit["Cell_type"]==cell_type)&(df_invit["Variant"]==variant)]
    max_WT = df_invit.loc[(df_invit["Plot"]==plot_num)&(df_invit["Cell_type"]==cell_type)&(df_invit["Variant"]=="WT"),"pSTAT"].max()
    fit, cov = curve_fit(hill_func, df_var["IL"], df_var["pSTAT"].values*100/(max_WT), bounds = ([0,df_var["IL"].min()/10], [df_var["pSTAT"].max()*100/(max_WT)+15, df_var["IL"].max()*10]))
    AMP_list.append(fit[0])
df_EC50_AMP_true["Amp"] = AMP_list
df_EC50_AMP_true = df_EC50_AMP_true.loc[df_EC50_AMP_true["Variant"] != "WT"]
identifier = df_EC50_AMP_true["Cell_type"]+"__"+df_EC50_AMP_true["Variant"]+"__"+df_EC50_AMP_true["STAT_type"]
df_EC50_AMP_true = df_EC50_AMP_true[["Amp"]].transpose()
df_EC50_AMP_true.columns = identifier

# pSTAT1 amplitude
df_EC50_AMP_S1 = df_EC50_AMP_true[[col for col in df_EC50_AMP_true.columns if "pSTAT1" in col]]
df_EC50_AMP_S1.columns = [col.replace("__pSTAT1","").replace("__"," \n") for col in df_EC50_AMP_S1.columns]

# pSTAT3 amplitude
df_EC50_AMP_S3 = df_EC50_AMP_true[[col for col in df_EC50_AMP_true.columns if "pSTAT3" in col]]
df_EC50_AMP_S3.columns = [col.replace("__pSTAT3","").replace("__"," \n") for col in df_EC50_AMP_S3.columns]
df_EC50_AMP_S3[df_EC50_AMP_S3<100] = 100

In [11]:
# Get data from simulations and isolate magnitude of the response
df_sim = pd.read_csv("results/fit_param/simulations_fit_IL10_RAp_ODE_MONO_fit_eIC.csv")
df_sim_S1 = df_sim.loc[df_sim["STAT_type"]=="pSTAT1"]
df_sim_S3 = df_sim.loc[df_sim["STAT_type"]=="pSTAT3"]
df_amp_S1 = pd.DataFrame()
for plot_num_OG in df_sim_S1["Plot OG"].drop_duplicates():
    df_amp_S1[df_sim_S1.loc[(df_sim_S1["Plot OG"]==plot_num_OG)&(df_sim_S1["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][0]+"__"+df_sim_S1.loc[(df_sim_S1["Plot OG"]==plot_num_OG)&(df_sim_S1["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][1]+"__"+df_sim_S1.loc[(df_sim_S1["Plot OG"]==plot_num_OG)&(df_sim_S1["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][2]] = [df_sim_S1.loc[(df_sim_S1["Plot"]==plot_num)&(df_sim_S1["Variant"] != "WT"),"Result"].max() for plot_num in df_sim_S1.loc[(df_sim_S1["Plot OG"]==plot_num_OG),"Plot"].drop_duplicates().values]
df_amp_S3 = pd.DataFrame()
for plot_num_OG in df_sim_S3["Plot OG"].drop_duplicates():
    df_amp_S3[df_sim_S3.loc[(df_sim_S3["Plot OG"]==plot_num_OG)&(df_sim_S3["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][0]+"__"+df_sim_S3.loc[(df_sim_S3["Plot OG"]==plot_num_OG)&(df_sim_S3["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][1]+"__"+df_sim_S3.loc[(df_sim_S3["Plot OG"]==plot_num_OG)&(df_sim_S3["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][2]] = [df_sim_S3.loc[(df_sim_S3["Plot"]==plot_num)&(df_sim_S3["Variant"] != "WT"),"Result"].max() for plot_num in df_sim_S3.loc[(df_sim_S3["Plot OG"]==plot_num_OG),"Plot"].drop_duplicates().values]

# Generate figure
fig, ax = plt.subplots(1, 1, figsize=(7, 8), dpi=400)
pos_base = np.array([0,0.9])
offset = 0.18
# Get the violin plots
quartile1, medians, quartile3 = np.percentile(df_amp_S1.T, [25, 50, 75], axis=1)
vp_S1 = ax.violinplot(df_amp_S1, positions=pos_base + offset,
              widths=0.3, showextrema=False)
for b in vp_S1['bodies']:
    b.set_facecolor("Darkred")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base + offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base + offset, df_amp_S1.mean(), marker='o', color='white', s=40, zorder=9)
ax.scatter(pos_base + offset,df_EC50_AMP_S1.values, s=250, marker='D', color="gold", edgecolor="black", linewidth=2, zorder=10)

quartile1, medians, quartile3 = np.percentile(df_amp_S3.T, [25, 50, 75], axis=1)
vp_S3 = ax.violinplot(df_amp_S3, positions=pos_base - offset,
              widths=0.3, showextrema=False)
for b in vp_S3['bodies']:
    b.set_facecolor("cornflowerblue")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base - offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base - offset, df_amp_S3.mean(), marker='o', color='white', s=40, zorder=9)
ax.scatter(pos_base - offset,df_EC50_AMP_S3.values, s=250, marker='D', color="gold", edgecolor="black", linewidth=2, zorder=10)

# Customize appearance
ax.set_xticks(pos_base)
ax.set_xticklabels(df_EC50_AMP_S3.columns, fontsize=18)
plt.ylabel("pSTAT (%)", fontsize=18)
legend_handles = [
    Patch.Patch(facecolor="cornflowerblue", label="pSTAT3"),
    Patch.Patch(facecolor="Darkred", label="pSTAT1")
]
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)
plt.ylabel("pSTAT (%)", fontsize=18)
ax.legend(handles=legend_handles, fontsize=18,loc="upper left")
plt.savefig('figures/fit_param/violin_amplitude_MONO_IL10_RAp_ODE.pdf', transparent=True, bbox_inches="tight", format="pdf")

### Receptor memory model assumming STAT1 can also be phophorylated in cis by Jak1 (STAT1 vs STAT3 amplitude response plot)

In [13]:
# Get experimental data on response amplitude in the different cells
df_EC50_AMP = pd.concat([pd.read_csv('data/signaling/IL10_EC50_AMP.tsv.gz', sep="\t", compression="gzip"), pd.read_csv('data/signaling/IL10_EC50_AMP_CD8_Gorby.tsv.gz', sep="\t", compression="gzip")], ignore_index=True)
variants = ["WT","Super-10","R5A11D"]
df_EC50_AMP = df_EC50_AMP.loc[(df_EC50_AMP["Variant"].isin(variants))&(df_EC50_AMP["Plot"].isin([1,2,8,9,12,13,14,15]))]
df_EC50_AMP = df_EC50_AMP.loc[df_EC50_AMP["Variant"] != "WT"]
identifier = df_EC50_AMP["Cell_type"]+"__"+df_EC50_AMP["Variant"]+"__"+df_EC50_AMP["STAT_type"]
df_EC50_AMP = df_EC50_AMP[["Amp"]].transpose()
df_EC50_AMP.columns = identifier
df_EC50_AMP.columns = [col.replace("MO.Mean","Monocytes") for col in df_EC50_AMP.columns]
df_EC50_AMP.columns = [col.replace("T4.Mean","CD4+ T cells") for col in df_EC50_AMP.columns]
df_EC50_AMP.columns = [col.replace("T8.Mean","CD8+ T cells") for col in df_EC50_AMP.columns]
df_EC50_AMP = df_EC50_AMP[['Monocytes__R5A11D__pSTAT3','Monocytes__R5A11D__pSTAT1','CD8+ T cells__R5A11D__pSTAT3','CD8+ T cells__R5A11D__pSTAT1','CD4+ T cells__Super-10__pSTAT3','CD8+ T cells__Super-10__pSTAT3','CD8+ T cells__Super-10__pSTAT1','CD4+ T cells__Super-10__pSTAT1']]

# Get distribution of simulated amplitude in the different cells
df_sim = pd.read_csv("results/fit_param/model_perturbations/simulations_fit_IL10_RAp_eIC_ODE_S1RA.csv")
df_sim = df_sim.loc[df_sim["Plot OG"].isin([1,2,8,9,12,13,14,15])]
for plot_num in df_sim["Plot"].drop_duplicates():
    df_sim.loc[(df_sim["Plot"]==plot_num),"Result"] = df_sim.loc[(df_sim["Plot"]==plot_num),"Result"]/df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"].max()*100
    df_amp = pd.DataFrame()
for plot_num_OG in df_sim["Plot OG"].drop_duplicates():
    df_amp[df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][0]+"__"+df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][1]+"__"+df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][2]] = [df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"] != "WT"),"Result"].max() for plot_num in df_sim.loc[(df_sim["Plot OG"]==plot_num_OG),"Plot"].drop_duplicates().values]
df_amp.columns = [col.replace("MO.Mean","Monocytes") for col in df_amp.columns]
df_amp.columns = [col.replace("T4.Mean","CD4+ T cells") for col in df_amp.columns]
df_amp.columns = [col.replace("T8.Mean","CD8+ T cells") for col in df_amp.columns]
df_amp = df_amp[['Monocytes__R5A11D__pSTAT3','Monocytes__R5A11D__pSTAT1','CD8+ T cells__R5A11D__pSTAT3','CD8+ T cells__R5A11D__pSTAT1','CD4+ T cells__Super-10__pSTAT3','CD8+ T cells__Super-10__pSTAT3','CD8+ T cells__Super-10__pSTAT1','CD4+ T cells__Super-10__pSTAT1']]

# pSTAT1 amplitude
df_amp_S1 = df_amp[[col for col in df_amp.columns if "pSTAT1" in col]]
df_amp_S1.columns = [col.replace("__pSTAT1","").replace("__"," \n") for col in df_amp_S1.columns]
df_EC50_AMP_S1 = df_EC50_AMP[[col for col in df_EC50_AMP.columns if "pSTAT1" in col]]
df_EC50_AMP_S1.columns = [col.replace("__pSTAT1","").replace("__"," \n") for col in df_EC50_AMP_S1.columns]

# pSTAT3 amplitude
df_amp_S3 = df_amp[[col for col in df_amp.columns if "pSTAT3" in col]]
df_amp_S3.columns = [col.replace("__pSTAT3","").replace("__"," \n") for col in df_amp_S3.columns]
df_EC50_AMP_S3 = df_EC50_AMP[[col for col in df_EC50_AMP.columns if "pSTAT3" in col]]
df_EC50_AMP_S3.columns = [col.replace("__pSTAT3","").replace("__"," \n") for col in df_EC50_AMP_S3.columns]
df_EC50_AMP_S3[df_EC50_AMP_S3<100] = 100

# Generate figure
fig, ax = plt.subplots(1, 1, figsize=(13, 8), dpi=400)
pos_base = np.arange(len(df_amp_S1.columns)) 
offset = 0.2
# Get the violin plots
quartile1, medians, quartile3 = np.percentile(df_amp_S1.T, [25, 50, 75], axis=1)
vp_S1 = ax.violinplot(df_amp_S1, positions=pos_base + offset,
              widths=0.3, showextrema=False)
for b in vp_S1['bodies']:
    b.set_facecolor("Darkred")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base + offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base + offset, df_amp_S1.mean(), marker='o', color='white', s=40, zorder=9)
ax.scatter(pos_base + offset,df_EC50_AMP_S1.values, s=250, marker='D', color="gold", edgecolor="black", linewidth=2, zorder=10)

quartile1, medians, quartile3 = np.percentile(df_amp_S3.T, [25, 50, 75], axis=1)
vp_S3 = ax.violinplot(df_amp_S3, positions=pos_base - offset,
              widths=0.3, showextrema=False)
for b in vp_S3['bodies']:
    b.set_facecolor("cornflowerblue")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base - offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base - offset, df_amp_S3.mean(), marker='o', color='white', s=40, zorder=9)
ax.scatter(pos_base - offset,df_EC50_AMP_S3.values, s=250, marker='D', color="gold", edgecolor="black", linewidth=2, zorder=10)

# Customize appearance
ax.set_xticks(pos_base)
ax.set_xticklabels(df_EC50_AMP_S3.columns, fontsize=18)
plt.ylabel("pSTAT (%)", fontsize=18)
legend_handles = [
    Patch.Patch(facecolor="cornflowerblue", label="pSTAT3"),
    Patch.Patch(facecolor="Darkred", label="pSTAT1")
]
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)
plt.ylabel("pSTAT (%)", fontsize=18)
ax.legend(handles=legend_handles, fontsize=18,loc="upper left")
plt.savefig('figures/fit_param/violin_amplitude_IL10_RAp_ODE_S1RA.pdf', transparent=True, bbox_inches="tight")
plt.close()

#### Same but with IL-10 monomer data

In [14]:
# Get amplitude of in vitro dose response curves
df_sim = pd.read_csv("results/fit_param/model_perturbations/simulations_fit_IL10_RAp_ODE_MONO_fit_eIC_S1RA.csv")
df_invit = pd.read_csv("data/signaling/IL10M_STAT_data_Gorby.tsv.gz", sep='\t', compression='gzip')
df_EC50_AMP = df_invit[['Plot', 'Cell_type', 'Variant', 'STAT_type']].drop_duplicates()
df_EC50_AMP.reset_index(drop=True,inplace=True)
AMP_list = []
for plot_num,cell_type,variant in [df_EC50_AMP.loc[i].values[:-1] for i in range(0,len(df_EC50_AMP))]:
    df_var = df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Cell_type"]==cell_type)&(df_sim["Variant"]==variant)]
    max_WT = df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Cell_type"]==cell_type)&(df_sim["Variant"]=="WT"),"Result"].max()
    fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values*100/(max_WT), bounds = ([0,df_var["IL0"].min()/10], [df_var["Result"].max()*100/(max_WT)+15, df_var["IL0"].max()*10]))
    AMP_list.append(fit[0])
df_EC50_AMP["Amp"] = AMP_list 
df_EC50_AMP = df_EC50_AMP.loc[df_EC50_AMP["Variant"] != "WT"]
identifier = df_EC50_AMP["Cell_type"]+"__"+df_EC50_AMP["Variant"]+"__"+df_EC50_AMP["STAT_type"]
df_EC50_AMP = df_EC50_AMP[["Amp"]].transpose()
df_EC50_AMP.columns = identifier

# Get amplitudes of experimental data
df_EC50_AMP_true = df_invit[['Plot', 'Cell_type', 'Variant', 'STAT_type']].drop_duplicates()
df_EC50_AMP_true.reset_index(drop=True,inplace=True)
AMP_list = []
for plot_num,cell_type,variant in [df_EC50_AMP_true.loc[i].values[:-1] for i in range(0,len(df_EC50_AMP_true))]:
    df_var = df_invit.loc[(df_invit["Plot"]==plot_num)&(df_invit["Cell_type"]==cell_type)&(df_invit["Variant"]==variant)]
    max_WT = df_invit.loc[(df_invit["Plot"]==plot_num)&(df_invit["Cell_type"]==cell_type)&(df_invit["Variant"]=="WT"),"pSTAT"].max()
    fit, cov = curve_fit(hill_func, df_var["IL"], df_var["pSTAT"].values*100/(max_WT), bounds = ([0,df_var["IL"].min()/10], [df_var["pSTAT"].max()*100/(max_WT)+15, df_var["IL"].max()*10]))
    AMP_list.append(fit[0])
df_EC50_AMP_true["Amp"] = AMP_list
df_EC50_AMP_true = df_EC50_AMP_true.loc[df_EC50_AMP_true["Variant"] != "WT"]
identifier = df_EC50_AMP_true["Cell_type"]+"__"+df_EC50_AMP_true["Variant"]+"__"+df_EC50_AMP_true["STAT_type"]
df_EC50_AMP_true = df_EC50_AMP_true[["Amp"]].transpose()
df_EC50_AMP_true.columns = identifier

# pSTAT1 amplitude
df_EC50_AMP_S1 = df_EC50_AMP_true[[col for col in df_EC50_AMP_true.columns if "pSTAT1" in col]]
df_EC50_AMP_S1.columns = [col.replace("__pSTAT1","").replace("__"," \n") for col in df_EC50_AMP_S1.columns]

# pSTAT3 amplitude
df_EC50_AMP_S3 = df_EC50_AMP_true[[col for col in df_EC50_AMP_true.columns if "pSTAT3" in col]]
df_EC50_AMP_S3.columns = [col.replace("__pSTAT3","").replace("__"," \n") for col in df_EC50_AMP_S3.columns]
df_EC50_AMP_S3[df_EC50_AMP_S3<100] = 100

In [15]:
# Get data from simulations and isolate magnitude of the response
df_sim = pd.read_csv("results/fit_param/model_perturbations/simulations_fit_IL10_RAp_ODE_MONO_fit_eIC_S1RA.csv")
df_sim_S1 = df_sim.loc[df_sim["STAT_type"]=="pSTAT1"]
df_sim_S3 = df_sim.loc[df_sim["STAT_type"]=="pSTAT3"]
df_amp_S1 = pd.DataFrame()
for plot_num_OG in df_sim_S1["Plot OG"].drop_duplicates():
    df_amp_S1[df_sim_S1.loc[(df_sim_S1["Plot OG"]==plot_num_OG)&(df_sim_S1["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][0]+"__"+df_sim_S1.loc[(df_sim_S1["Plot OG"]==plot_num_OG)&(df_sim_S1["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][1]+"__"+df_sim_S1.loc[(df_sim_S1["Plot OG"]==plot_num_OG)&(df_sim_S1["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][2]] = [df_sim_S1.loc[(df_sim_S1["Plot"]==plot_num)&(df_sim_S1["Variant"] != "WT"),"Result"].max() for plot_num in df_sim_S1.loc[(df_sim_S1["Plot OG"]==plot_num_OG),"Plot"].drop_duplicates().values]
df_amp_S3 = pd.DataFrame()
for plot_num_OG in df_sim_S3["Plot OG"].drop_duplicates():
    df_amp_S3[df_sim_S3.loc[(df_sim_S3["Plot OG"]==plot_num_OG)&(df_sim_S3["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][0]+"__"+df_sim_S3.loc[(df_sim_S3["Plot OG"]==plot_num_OG)&(df_sim_S3["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][1]+"__"+df_sim_S3.loc[(df_sim_S3["Plot OG"]==plot_num_OG)&(df_sim_S3["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][2]] = [df_sim_S3.loc[(df_sim_S3["Plot"]==plot_num)&(df_sim_S3["Variant"] != "WT"),"Result"].max() for plot_num in df_sim_S3.loc[(df_sim_S3["Plot OG"]==plot_num_OG),"Plot"].drop_duplicates().values]

# Generate figure
fig, ax = plt.subplots(1, 1, figsize=(7, 8), dpi=400)
pos_base = np.array([0,0.9])
offset = 0.18
# Get the violin plots
quartile1, medians, quartile3 = np.percentile(df_amp_S1.T, [25, 50, 75], axis=1)
vp_S1 = ax.violinplot(df_amp_S1, positions=pos_base + offset,
              widths=0.3, showextrema=False)
for b in vp_S1['bodies']:
    b.set_facecolor("Darkred")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base + offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base + offset, df_amp_S1.mean(), marker='o', color='white', s=40, zorder=9)
ax.scatter(pos_base + offset,df_EC50_AMP_S1.values, s=250, marker='D', color="gold", edgecolor="black", linewidth=2, zorder=10)

quartile1, medians, quartile3 = np.percentile(df_amp_S3.T, [25, 50, 75], axis=1)
vp_S3 = ax.violinplot(df_amp_S3, positions=pos_base - offset,
              widths=0.3, showextrema=False)
for b in vp_S3['bodies']:
    b.set_facecolor("cornflowerblue")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base - offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base - offset, df_amp_S3.mean(), marker='o', color='white', s=40, zorder=9)
ax.scatter(pos_base - offset,df_EC50_AMP_S3.values, s=250, marker='D', color="gold", edgecolor="black", linewidth=2, zorder=10)

# Customize appearance
ax.set_xticks(pos_base)
ax.set_xticklabels(df_EC50_AMP_S3.columns, fontsize=18)
plt.ylabel("pSTAT (%)", fontsize=18)
legend_handles = [
    Patch.Patch(facecolor="cornflowerblue", label="pSTAT3"),
    Patch.Patch(facecolor="Darkred", label="pSTAT1")
]
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)
plt.ylabel("pSTAT (%)", fontsize=18)
ax.legend(handles=legend_handles, fontsize=18,loc="upper left")
plt.savefig('figures/fit_param/violin_amplitude_MONO_IL10_RAp_ODE_S1RA.pdf', transparent=True, bbox_inches="tight", format="pdf")
plt.close()

### Explaining signaling decay at high IL-10 concentrations through IL-10RA binding producing half-saturation (dose-response curves)

In [16]:
df_invit = pd.concat([pd.read_csv('data/signaling/IL10_STAT_data.tsv.gz', sep="\t", compression="gzip"), pd.read_csv('data/signaling/IL10_STAT_data_CD8_Gorby.tsv.gz', sep="\t", compression="gzip")], ignore_index=True)
df_invit.loc[len(df_invit)] = [12, 'T8.Mean', 'WT', 'pSTAT1', 1e-7, 100.297619047619]
df_invit.loc[len(df_invit)] = [12, 'T8.Mean', 'Super-10', 'pSTAT1', 1e-7, 146.130952380952]
df_invit.loc[len(df_invit)] = [13, 'T4.Mean', 'WT', 'pSTAT1', 1e-7, 100.595238095238]
df_invit.loc[len(df_invit)] = [13, 'T4.Mean', 'Super-10', 'pSTAT1', 1e-7, 136.309523809524]
df_sim = pd.read_csv("results/fit_param/model_perturbations/simulations_fit_IL10_RAp_eIC_ODE_RAsat.csv")
# Normalize data to the maximum WT response
for plot_num in df_sim["Plot"].drop_duplicates():
    df_sim.loc[(df_sim["Plot"]==plot_num),"Result"] = df_sim.loc[(df_sim["Plot"]==plot_num),"Result"]/df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"].max()*100
# Plot dose-response curves
for plot_num in df_sim["Plot OG"].drop_duplicates():
    df_invit_plot = df_invit.loc[df_invit["Plot"]==plot_num]

    # First plot WT
    variant = "WT"
    variant_list = df_sim.loc[df_sim["Plot OG"]==plot_num,"Variant"].drop_duplicates().to_list()
    variant_list.remove("WT")
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    IL = df_sim.loc[(df_sim["Plot OG"]==plot_num),"IL0"].drop_duplicates().values
    responses = df_sim.loc[(df_sim["Plot OG"]==plot_num)&(df_sim["Variant"]==variant)][["Plot","IL0","Result"]].pivot(index='IL0', columns='Plot', values='Result').transpose().values
    original_response = responses[0]
    low_response = np.percentile(responses, 5, axis=0)
    high_response = np.percentile(responses, 95, axis=0)
    ax.fill_between(IL, low_response, high_response, color="lightgreen", alpha=0.5)
    ax.plot(IL, original_response, linewidth=7.5, label=variant,color="darkgreen", linestyle="dashed")
    ax.scatter(df_invit_plot.loc[df_invit_plot["Variant"]==variant]["IL"].values, df_invit_plot.loc[df_invit_plot["Variant"]==variant]["pSTAT"].values, s=200, linewidth=2, color="forestgreen",marker='*')

    i=0
    # Then plot the other variants
    for variant in variant_list:
        responses = df_sim.loc[(df_sim["Plot OG"]==plot_num)&(df_sim["Variant"]==variant)][["Plot","IL0","Result"]].pivot(index='IL0', columns='Plot', values='Result').transpose().values
        original_response = responses[0]
        low_response = np.percentile(responses, 5, axis=0)
        high_response = np.percentile(responses, 95, axis=0)
        ax.fill_between(IL, low_response, high_response, color=color_area[i], alpha=0.4)
        ax.plot(IL, original_response, linewidth=7.5, label=variant,color=color_sim[i], linestyle=linestyles[i])
        ax.scatter(df_invit_plot.loc[df_invit_plot["Variant"]==variant]["IL"].values, df_invit_plot.loc[df_invit_plot["Variant"]==variant]["pSTAT"].values, s=200, linewidth=2, color=color_invit[i],marker=markers[i])
        i += 1
    plt.legend(loc="upper left", fontsize=20)
    plt.xscale('log')
    ax.set_xticks(10**np.arange(round(np.log10(df_sim.loc[df_sim["Plot OG"]==plot_num,"IL0"].min()),0)+1,round(np.log10(df_sim.loc[df_sim["Plot OG"]==plot_num,"IL0"].max()),0)+1,2))
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    cell = df_sim.loc[df_sim["Plot OG"]==plot_num,"Cell_type"].values[0]
    plt.title(cell, fontsize=25)
    plt.ylabel(list(df_sim.loc[df_sim["Plot OG"]==plot_num,"STAT_type"])[0] +' (%)', fontsize=25)
    plt.xlabel('IL-10 (M)', fontsize=25)
    plt.savefig('figures/fit_param/IL10_RAp_ODE/RAsat/Plot_n'+str(df_sim.loc[df_sim["Plot OG"]==plot_num,"Plot"].values[0])+'_cell_'+df_sim.loc[df_sim["Plot OG"]==plot_num,"Cell_type"].values[0]+'.pdf', transparent=True, bbox_inches="tight")
    plt.close()

### Explaining signaling decay at high IL-10 concentrations through higher IL-10RB binding (dose-response curves)

In [17]:
df_invit = pd.concat([pd.read_csv('data/signaling/IL10_STAT_data.tsv.gz', sep="\t", compression="gzip"), pd.read_csv('data/signaling/IL10_STAT_data_CD8_Gorby.tsv.gz', sep="\t", compression="gzip")], ignore_index=True)
df_invit.loc[len(df_invit)] = [12, 'T8.Mean', 'WT', 'pSTAT1', 1e-7, 100.297619047619]
df_invit.loc[len(df_invit)] = [12, 'T8.Mean', 'Super-10', 'pSTAT1', 1e-7, 146.130952380952]
df_invit.loc[len(df_invit)] = [13, 'T4.Mean', 'WT', 'pSTAT1', 1e-7, 100.595238095238]
df_invit.loc[len(df_invit)] = [13, 'T4.Mean', 'Super-10', 'pSTAT1', 1e-7, 136.309523809524]
df_sim = pd.read_csv("results/fit_param/model_perturbations/simulations_fit_IL10_RAp_eIC_ODE_RBsat.csv")
# Normalize data to the maximum WT response
for plot_num in df_sim["Plot"].drop_duplicates():
    df_sim.loc[(df_sim["Plot"]==plot_num),"Result"] = df_sim.loc[(df_sim["Plot"]==plot_num),"Result"]/df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"].max()*100
# Plot dose-response curves
for plot_num in df_sim["Plot OG"].drop_duplicates():
    df_invit_plot = df_invit.loc[df_invit["Plot"]==plot_num]

    # First plot WT
    variant = "WT"
    variant_list = df_sim.loc[df_sim["Plot OG"]==plot_num,"Variant"].drop_duplicates().to_list()
    variant_list.remove("WT")
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    IL = df_sim.loc[(df_sim["Plot OG"]==plot_num),"IL0"].drop_duplicates().values
    responses = df_sim.loc[(df_sim["Plot OG"]==plot_num)&(df_sim["Variant"]==variant)][["Plot","IL0","Result"]].pivot(index='IL0', columns='Plot', values='Result').transpose().values
    original_response = responses[0]
    low_response = np.percentile(responses, 5, axis=0)
    high_response = np.percentile(responses, 95, axis=0)
    ax.fill_between(IL, low_response, high_response, color="lightgreen", alpha=0.5)
    ax.plot(IL, original_response, linewidth=7.5, label=variant,color="darkgreen", linestyle="dashed")
    ax.scatter(df_invit_plot.loc[df_invit_plot["Variant"]==variant]["IL"].values, df_invit_plot.loc[df_invit_plot["Variant"]==variant]["pSTAT"].values, s=200, linewidth=2, color="forestgreen",marker='*')

    i=0
    # Then plot the other variants
    for variant in variant_list:
        responses = df_sim.loc[(df_sim["Plot OG"]==plot_num)&(df_sim["Variant"]==variant)][["Plot","IL0","Result"]].pivot(index='IL0', columns='Plot', values='Result').transpose().values
        original_response = responses[0]
        low_response = np.percentile(responses, 5, axis=0)
        high_response = np.percentile(responses, 95, axis=0)
        ax.fill_between(IL, low_response, high_response, color=color_area[i], alpha=0.4)
        ax.plot(IL, original_response, linewidth=7.5, label=variant,color=color_sim[i], linestyle=linestyles[i])
        ax.scatter(df_invit_plot.loc[df_invit_plot["Variant"]==variant]["IL"].values, df_invit_plot.loc[df_invit_plot["Variant"]==variant]["pSTAT"].values, s=200, linewidth=2, color=color_invit[i],marker=markers[i])
        i += 1
    plt.legend(loc="upper left", fontsize=20)
    plt.xscale('log')
    ax.set_xticks(10**np.arange(round(np.log10(df_sim.loc[df_sim["Plot OG"]==plot_num,"IL0"].min()),0)+1,round(np.log10(df_sim.loc[df_sim["Plot OG"]==plot_num,"IL0"].max()),0)+1,2))
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    cell = df_sim.loc[df_sim["Plot OG"]==plot_num,"Cell_type"].values[0]
    plt.title(cell, fontsize=25)
    plt.ylabel(list(df_sim.loc[df_sim["Plot OG"]==plot_num,"STAT_type"])[0] +' (%)', fontsize=25)
    plt.xlabel('IL-10 (M)', fontsize=25)
    plt.savefig('figures/fit_param/IL10_RAp_ODE/RBsat/Plot_n'+str(df_sim.loc[df_sim["Plot OG"]==plot_num,"Plot"].values[0])+'_cell_'+df_sim.loc[df_sim["Plot OG"]==plot_num,"Cell_type"].values[0]+'.pdf', transparent=True, bbox_inches="tight")
    plt.close()

## Change the phosphorylation rate and see its effect in an IL-10RB mutant

In [18]:
# Get simulation data
df_sim = pd.read_csv("results/fit_param/model_perturbations/simulations_fit_IL10_RAp_eIC_ODE_phos.csv")
df_sim = df_sim.loc[df_sim["Cell_type"]=="T8.Mean"]
df_sim["Plot"] = df_sim["Plot"].astype("str")+df_sim["Perturbation"]
df_sim["Plot OG"] = df_sim["Plot OG"].astype("str")+df_sim["Perturbation"]
df_sim["Plot OG"] = [{
    '3Normal':1, 
    '4Normal':2, 
    '3More Dephosphorylation':3, 
    '4More Dephosphorylation':4,
    '3Less Dephosphorylation':5,
    '4Less Dephosphorylation':6
}[num] for num in df_sim["Plot OG"].values]
df_sim["Cell_type"] = df_sim["Perturbation"]

# Get amplitudes
for plot_num in df_sim["Plot"].drop_duplicates():
    df_sim.loc[(df_sim["Plot"]==plot_num),"Result"] = df_sim.loc[(df_sim["Plot"]==plot_num),"Result"]/df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"].max()*100
df_amp = pd.DataFrame()
for plot_num_OG in df_sim["Plot OG"].drop_duplicates():
    df_amp[df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][0]+"__"+df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][1]+"__"+df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] != "WT")][["Cell_type","Variant","STAT_type"]].values[0][2]] = [df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"] != "WT"),"Result"].max() for plot_num in df_sim.loc[(df_sim["Plot OG"]==plot_num_OG),"Plot"].drop_duplicates().values]
df_amp = df_amp.mask(df_amp < 95)

# pSTAT1 amplitude
df_amp_S1 = df_amp[[col for col in df_amp.columns if "pSTAT1" in col]]
df_amp_S1.columns = [col.replace("__pSTAT1","").replace("__"," \n") for col in df_amp_S1.columns]

# pSTAT3 amplitude
df_amp_S3 = df_amp[[col for col in df_amp.columns if "pSTAT3" in col]]
df_amp_S3.columns = [col.replace("__pSTAT3","").replace("__"," \n") for col in df_amp_S3.columns]

# Generate figure
fig, ax = plt.subplots(1, 1, figsize=(13, 8), dpi=400)
pos_base = np.arange(len(df_amp_S1.columns)) 
offset = 0.2
# Get the violin plots
quartile1, medians, quartile3 = np.percentile(df_amp_S1.T, [25, 50, 75], axis=1)
vp_S1 = ax.violinplot(df_amp_S1, positions=pos_base + offset,
              widths=0.3, showextrema=False)
for b in vp_S1['bodies']:
    b.set_facecolor("Darkred")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base + offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base + offset, df_amp_S1.mean(), marker='o', color='white', s=40, zorder=9)

quartile1, medians, quartile3 = np.percentile(df_amp_S3.T, [25, 50, 75], axis=1)
vp_S3 = ax.violinplot([df_amp_S3[col].dropna().values for col in df_amp_S3.columns], positions=pos_base - offset,
              widths=0.3, showextrema=False)
for b in vp_S3['bodies']:
    b.set_facecolor("cornflowerblue")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base - offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base - offset, df_amp_S3.mean(), marker='o', color='white', s=40, zorder=9)

# Customize appearance
ax.set_xticks(pos_base)
ax.set_xticklabels(['Normal','More\nDephosphorylation','Less\nDephosphorylation'], fontsize=18)
plt.ylabel("pSTAT (%)", fontsize=18)
legend_handles = [
    Patch.Patch(facecolor="cornflowerblue", label="pSTAT3"),
    Patch.Patch(facecolor="Darkred", label="pSTAT1")
]
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)
plt.ylabel("pSTAT (%)", fontsize=18)
ax.legend(handles=legend_handles, fontsize=18,loc="upper left")
plt.savefig('figures/fit_param/IL10_RAp_ODE/phos/violin_amplitude_IL10_RAp_eIC_ODE_phos.pdf', transparent=True, bbox_inches="tight")
plt.close()

## See the effects of chaging the Kon vs Koff in an IL-10RB mutant

In [19]:
# Get simulation data
df_sim = pd.read_csv("results/fit_param/model_perturbations/simulations_fit_IL10_RAp_eIC_ODE_RB_Kon_off.csv")
df_sim = df_sim.loc[df_sim["Cell_type"]=="ACH-000786"]

# Get amplitudes
for plot_num in df_sim["Plot"].drop_duplicates():
    df_sim.loc[(df_sim["Plot"]==plot_num),"Result"] = df_sim.loc[(df_sim["Plot"]==plot_num),"Result"]/df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"].max()*100
df_amp = pd.DataFrame()
for plot_num_OG in df_sim["Plot OG"].drop_duplicates():
    for variant in df_sim.loc[df_sim["Variant"]!="WT","Variant"].drop_duplicates():
        df_amp[df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] == variant)][["Cell_type","Variant","STAT_type"]].values[0][0]+"__"+df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] == variant)][["Cell_type","Variant","STAT_type"]].values[0][1]+"__"+df_sim.loc[(df_sim["Plot OG"]==plot_num_OG)&(df_sim["Variant"] == variant)][["Cell_type","Variant","STAT_type"]].values[0][2]] = [df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"] == variant),"Result"].max() for plot_num in df_sim.loc[(df_sim["Plot OG"]==plot_num_OG),"Plot"].drop_duplicates().values]

# pSTAT1 amplitude
df_amp_S1 = df_amp[[col for col in df_amp.columns if "pSTAT1" in col]]
df_amp_S1.columns = [col.replace("__pSTAT1","").replace("__"," \n") for col in df_amp_S1.columns]

# pSTAT3 amplitude
df_amp_S3 = df_amp[[col for col in df_amp.columns if "pSTAT3" in col]]
df_amp_S3.columns = [col.replace("__pSTAT3","").replace("__"," \n") for col in df_amp_S3.columns]

# Generate figure
fig, ax = plt.subplots(1, 1, figsize=(9, 8), dpi=400)
pos_base = np.arange(len(df_amp_S1.columns)) 
offset = 0.2
# Get the violin plots
quartile1, medians, quartile3 = np.percentile(df_amp_S1.T, [25, 50, 75], axis=1)
vp_S1 = ax.violinplot(df_amp_S1, positions=pos_base + offset,
              widths=0.3, showextrema=False)
for b in vp_S1['bodies']:
    b.set_facecolor("Darkred")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base + offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base + offset, df_amp_S1.mean(), marker='o', color='white', s=40, zorder=9)

quartile1, medians, quartile3 = np.percentile(df_amp_S3.T, [25, 50, 75], axis=1)
vp_S3 = ax.violinplot([df_amp_S3[col].dropna().values for col in df_amp_S3.columns], positions=pos_base - offset,
              widths=0.3, showextrema=False)
for b in vp_S3['bodies']:
    b.set_facecolor("cornflowerblue")
    b.set_edgecolor("black")
    b.set_alpha(1)
ax.vlines(pos_base - offset, quartile1, quartile3, color='k', linestyle='-', lw=5)
ax.scatter(pos_base - offset, df_amp_S3.mean(), marker='o', color='white', s=40, zorder=9)

# Customize appearance
ax.set_xticks(pos_base)
ax.set_xticklabels(['IL-10RB mutant\nKoff','IL-10RB mutant\nKon'], fontsize=18)
plt.ylabel("pSTAT (%)", fontsize=18)
legend_handles = [
    Patch.Patch(facecolor="cornflowerblue", label="pSTAT3"),
    Patch.Patch(facecolor="Darkred", label="pSTAT1")
]
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)
plt.ylabel("pSTAT (%)", fontsize=18)
ax.legend(handles=legend_handles, fontsize=18,loc="upper left")
plt.savefig('figures/fit_param/IL10_RAp_ODE/RB_Kon_off/violin_amplitude_IL10_RAp_eIC_ODE_RB_Kon_off.pdf', transparent=True, bbox_inches="tight")
plt.close()